# 毎日新聞の記事検索結果を取得する

毎日新聞デジタルで「熊本地震」を検索し、2026年7月27日以降に公開された記事のタイトル・本文・公開日時・URLを取得します。

- 毎日IDでログインし、契約上閲覧できる記事本文を取得します。アクセス制限の回避は行いません。
- 短時間に大量アクセスしないよう、待機時間と取得件数の上限を設けています。
- 実行前に毎日新聞の利用規約・著作権・robots.txtを確認し、取得データは許可された範囲で利用してください。


In [1]:
# 初回だけ実行してください
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "requests", "beautifulsoup4", "pandas", "playwright"
])



[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


0

In [2]:
import json
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://mainichi.jp"
SEARCH_URL = f"{BASE_URL}/search"
LOGIN_URL = f"{BASE_URL}/signup/accounts/free/login/"
KEYWORD = "熊本地震"
START_DATE = pd.Timestamp("2026-07-27", tz="Asia/Tokyo")

# Noneなら検索結果の最終ページまで、全記事を取得する。
MAX_PAGES = None
MAX_ARTICLES = None
REQUEST_INTERVAL = 1.5
TIMEOUT = 20

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "ja,en-US;q=0.8,en;q=0.6",
})


## ログイン情報の入力

IDとパスワードは変数にだけ保持され、ノートブックファイルには保存されません。パスワードは入力中も画面に表示されません。


In [3]:
from getpass import getpass

MAINICHI_ID = input("毎日ID（メールアドレス）: ").strip()
MAINICHI_PASSWORD = getpass("パスワード: ")


In [4]:
from playwright.async_api import TimeoutError as PlaywrightTimeoutError
from playwright.async_api import async_playwright


async def submit_field_form(field):
    """ボタンのDOM構造に依存せず、入力欄が属するフォームを送信する。"""
    form = field.locator("xpath=ancestor::form[1]")
    if await form.count() > 0:
        # evaluate自体がページ遷移に巻き込まれないよう、送信を次のイベントへ予約する。
        await form.evaluate(
            "form => { window.setTimeout(() => form.requestSubmit(), 0); }"
        )
    else:
        await field.press("Enter")


def is_mainichi_login_page(url):
    """毎日IDのログイン・登録途中ページかを判定する。"""
    lowered = url.lower().rstrip("/")
    return (
        "/signup/accounts/free/login" in lowered
        or "/signup/accounts/login" in lowered
    )


def find_mainichi_logged_in_page(context, *, allow_general_page=False):
    """全タブから、ログイン完了後に表示できる毎日新聞ページを探す。"""
    pages = list(reversed(context.pages))
    for candidate in pages:
        url = candidate.url.lower()
        # 末尾スラッシュの有無やマイページ配下のURL変更を許容する。
        if "/signup/accounts/mypage" in url and not is_mainichi_login_page(url):
            return candidate
    if allow_general_page:
        for candidate in pages:
            url = candidate.url.lower()
            if (
                (url == BASE_URL or url.startswith(BASE_URL + "/"))
                and not is_mainichi_login_page(url)
                and url != "about:blank"
            ):
                return candidate
    return None


async def login_to_mainichi(requests_session, mainichi_id, password):
    """Chromeで毎日IDにログインし、Cookieをrequests.Sessionへ渡す。"""
    if not mainichi_id or not password:
        raise ValueError("毎日IDとパスワードを入力してください。")

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(channel="chrome", headless=False)
        context = await browser.new_context(locale="ja-JP")
        page = await context.new_page()
        await page.goto(LOGIN_URL, wait_until="domcontentloaded", timeout=60_000)

        username = page.locator(
            'input[autocomplete="username"], input[type="email"], '
            'input[name*="mail" i], input[type="text"]'
        ).first
        await page.wait_for_timeout(1_500)
        if await username.count() > 0 and await username.is_visible():
            await username.fill(mainichi_id)

            password_input = page.locator('input[type="password"]').first
            if not await password_input.is_visible():
                await submit_field_form(username)
                try:
                    await password_input.wait_for(state="visible", timeout=10_000)
                except PlaywrightTimeoutError:
                    print("パスワード入力以降はブラウザで手動操作してください。")

            if await password_input.count() > 0 and await password_input.is_visible():
                await password_input.fill(password)
                await submit_field_form(password_input)
                await page.wait_for_timeout(2_000)
        else:
            print(
                "ログイン入力欄を自動検出できませんでした。"
                "ブラウザ上で毎日IDとパスワードを手動入力してください。"
            )

        login_confirmed = False
        for _ in range(20):
            logged_in_page = find_mainichi_logged_in_page(context)
            if logged_in_page is not None:
                page = logged_in_page
                login_confirmed = True
                print(f"毎日IDのログイン完了を確認しました: {page.url}")
                break
            await page.wait_for_timeout(500)

        if not login_confirmed:
            print(
                "ブラウザ上のプラン選択、オプション確認、追加認証などを完了し、"
                "毎日新聞のマイページ、ホームまたは記事ページまで進んでください。"
            )
            while True:
                confirmation = input(
                    "ブラウザでログイン完了を確認したら DONE と入力してください: "
                ).strip().upper()
                if confirmation != "DONE":
                    print("まだブラウザは閉じません。完了後に DONE と入力してください。")
                    continue
                logged_in_page = find_mainichi_logged_in_page(context, allow_general_page=True)
                if logged_in_page is None:
                    # ログイン画面に留まって見える場合も、同じブラウザコンテキストで
                    # マイページへ遷移して、認証Cookieによるリダイレクト結果を確認する。
                    probe_page = next(
                        (candidate for candidate in reversed(context.pages) if not candidate.is_closed()),
                        None,
                    )
                    if probe_page is not None:
                        try:
                            await probe_page.goto(
                                f"{BASE_URL}/signup/accounts/mypage/",
                                wait_until="domcontentloaded",
                                timeout=30_000,
                            )
                        except PlaywrightTimeoutError:
                            pass
                        logged_in_page = find_mainichi_logged_in_page(
                            context, allow_general_page=True
                        )
                if logged_in_page is None:
                    open_urls = "\n".join(
                        f"- {candidate.url}" for candidate in context.pages
                    )
                    print(
                        f"ログイン完了ページをまだ確認できません。開いているページ:\n"
                        f"{open_urls}\n"
                        "ブラウザ側の画面操作を完了してから、もう一度 DONE と入力してください。"
                    )
                    continue
                page = logged_in_page
                print(f"毎日IDのログイン完了を確認しました: {page.url}")
                break

        cookies = await context.cookies(BASE_URL)
        for cookie in cookies:
            requests_session.cookies.set(
                cookie["name"],
                cookie["value"],
                domain=cookie.get("domain"),
                path=cookie.get("path", "/"),
            )
        await browser.close()

    if not cookies:
        raise RuntimeError("認証Cookieを取得できませんでした。")
    print("毎日IDのログインCookieを取得しました。")


await login_to_mainichi(session, MAINICHI_ID, MAINICHI_PASSWORD)
del MAINICHI_ID, MAINICHI_PASSWORD


ブラウザ上のプラン選択、オプション確認、追加認証などを完了し、毎日新聞のマイページ、ホームまたは記事ページまで進んでください。
毎日IDのログイン完了を確認しました: https://mainichi.jp/signup/accounts/mypage/
毎日IDのログインCookieを取得しました。


In [5]:
def get_soup(url, *, params=None):
    response = session.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    return BeautifulSoup(response.content, "html.parser")


def parse_search_page(soup):
    """検索結果1ページの記事一覧と次ページURLを返す。"""
    rows = []
    for item in soup.select("ul.articlelist > li"):
        link = item.select_one("a[href]")
        title_node = item.select_one(".articlelist-title")
        if link is None or title_node is None:
            continue

        url = urljoin(BASE_URL, link.get("href", ""))
        # 記事詳細ページのみを対象にする。
        if "/articles/" not in url:
            continue

        date_node = item.select_one(".articletag-date")
        rows.append({
            "title": title_node.get_text(" ", strip=True),
            "published_at": date_node.get_text(" ", strip=True) if date_node else None,
            "url": url,
        })

    next_node = soup.select_one("#nexturl[data-searchnexturl]")
    next_url = (
        urljoin(BASE_URL, next_node.get("data-searchnexturl", ""))
        if next_node else None
    )
    return rows, next_url


def search_articles(keyword, start_date, max_pages=None, max_articles=None):
    """HTML内の次ページURLをたどり、指定日以降の検索結果を返す。"""
    found = []
    seen = set()
    visited_pages = set()
    next_url = SEARCH_URL
    params = {"q": keyword, "op": "and", "s": "dd"}

    page_number = 0
    while next_url:
        if max_pages is not None and page_number >= max_pages:
            break
        # サイト側が同じnext URLを返し続けた場合の無限ループを防ぐ。
        page_key = (next_url, tuple(sorted(params.items())) if page_number == 0 else None)
        if page_key in visited_pages:
            print(f"同じ検索ページが再度返されたため終了します: {next_url}")
            break
        visited_pages.add(page_key)

        soup = get_soup(next_url, params=params if page_number == 0 else None)
        page_rows, next_url = parse_search_page(soup)
        page_number += 1
        new_count = 0
        reached_older_articles = False

        for row in page_rows:
            if row["url"] in seen:
                continue
            seen.add(row["url"])
            new_count += 1
            published_at = pd.to_datetime(row["published_at"], errors="coerce")
            if pd.isna(published_at):
                continue
            if published_at.tzinfo is None:
                published_at = published_at.tz_localize(start_date.tz)
            else:
                published_at = published_at.tz_convert(start_date.tz)
            if published_at < start_date:
                reached_older_articles = True
                continue
            found.append(row)
            if max_articles is not None and len(found) >= max_articles:
                return found

        print(f"検索ページ {page_number}: {start_date.date()} 以降 累計 {len(found)}件")
        if reached_older_articles:
            print(f"{start_date.date()} より前の記事に達したため検索を終了します。")
            break
        if not next_url or new_count == 0:
            break
        time.sleep(REQUEST_INTERVAL)

    return found


def news_article_json_ld(soup):
    """JSON-LDからNewsArticleメタデータを探す。"""
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue
        candidates = data if isinstance(data, list) else [data]
        for candidate in candidates:
            if isinstance(candidate, dict) and candidate.get("@type") in {
                "NewsArticle", "Article"
            }:
                return candidate
    return {}


def parse_article_page(soup, search_row):
    metadata = news_article_json_ld(soup)
    body_node = soup.select_one("#articledetail-body")
    paragraphs = []
    if body_node:
        for node in body_node.select("p"):
            text = node.get_text(" ", strip=True)
            if text:
                paragraphs.append(text)

    body = "\n\n".join(dict.fromkeys(paragraphs))
    body_is_excerpt = soup.select_one("#articledetail-pay") is not None

    # 本文要素がない場合だけ公開メタデータの概要を使用する。
    if not body:
        body = str(metadata.get("description") or "").strip()
        body_is_excerpt = bool(body)

    title = str(metadata.get("headline") or search_row["title"])
    if title.endswith(" - 毎日新聞"):
        title = title.removesuffix(" - 毎日新聞")

    return {
        "title": title,
        "body": body,
        "published_at": metadata.get("datePublished") or search_row["published_at"],
        "url": search_row["url"],
        "body_is_excerpt": body_is_excerpt,
    }


def collect_articles(keyword, start_date, max_pages=None, max_articles=None):
    search_rows = search_articles(keyword, start_date, max_pages, max_articles)
    print(f"検索結果から {len(search_rows)} 件のURLを取得しました。")

    articles = []
    for index, row in enumerate(search_rows, start=1):
        print(f"[{index}/{len(search_rows)}] {row['title']}")
        try:
            soup = get_soup(row["url"])
            articles.append(parse_article_page(soup, row))
        except requests.RequestException as exc:
            articles.append({
                **row,
                "body": "",
                "body_is_excerpt": False,
                "error": str(exc),
            })
        if index < len(search_rows):
            time.sleep(REQUEST_INTERVAL)

    return articles


In [6]:
articles = collect_articles(
    KEYWORD,
    START_DATE,
    max_pages=MAX_PAGES,
    max_articles=MAX_ARTICLES,
)

df = pd.DataFrame(articles)
if not df.empty:
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df = df.sort_values("published_at", ascending=False, na_position="last")
df = df.reset_index(drop=True)
df.insert(0, "ID", range(1, len(df) + 1))

df


検索ページ 1: 新規 20件 / 累計 20件
検索ページ 2: 新規 20件 / 累計 40件
検索ページ 3: 新規 20件 / 累計 60件
検索ページ 4: 新規 20件 / 累計 80件
検索ページ 5: 新規 20件 / 累計 100件
検索ページ 6: 新規 20件 / 累計 120件
検索ページ 7: 新規 19件 / 累計 139件
検索ページ 8: 新規 20件 / 累計 159件
検索ページ 9: 新規 20件 / 累計 179件
検索ページ 10: 新規 19件 / 累計 198件
検索ページ 11: 新規 20件 / 累計 218件
検索ページ 12: 新規 19件 / 累計 237件
検索ページ 13: 新規 20件 / 累計 257件
検索ページ 14: 新規 19件 / 累計 276件
検索ページ 15: 新規 20件 / 累計 296件
検索ページ 16: 新規 20件 / 累計 316件
検索ページ 17: 新規 20件 / 累計 336件
検索ページ 18: 新規 20件 / 累計 356件
検索ページ 19: 新規 20件 / 累計 376件
検索ページ 20: 新規 20件 / 累計 396件
検索ページ 21: 新規 20件 / 累計 416件
検索ページ 22: 新規 20件 / 累計 436件
検索ページ 23: 新規 19件 / 累計 455件
検索ページ 24: 新規 20件 / 累計 475件
検索ページ 25: 新規 20件 / 累計 495件
検索ページ 26: 新規 20件 / 累計 515件
検索ページ 27: 新規 20件 / 累計 535件
検索ページ 28: 新規 20件 / 累計 555件
検索ページ 29: 新規 20件 / 累計 575件
検索ページ 30: 新規 20件 / 累計 595件
検索ページ 31: 新規 20件 / 累計 615件
検索ページ 32: 新規 20件 / 累計 635件
検索ページ 33: 新規 20件 / 累計 655件
検索ページ 34: 新規 20件 / 累計 675件
検索ページ 35: 新規 20件 / 累計 695件
検索ページ 36: 新規 20件 / 累計 715件
検索ページ 37: 新規 20件 / 累計 735件
検索ページ 38: 新規 2

,ID,title,body,published_at,url,body_is_excerpt,error
0,1,東京都、23区の大学定員規制の撤廃求める 国と2回目の協議会,国と東京都の協議会の会合が30日、首相官邸で開かれた。都側は改めて地方交付税制度の問題点を指...,2026-07-30 21:27:57+09:00,https://mainichi.jp/articles/20260730/k00/00m/...,False,NaN
1,2,ストレスためた幹部→職員にあたる負の連鎖も 横浜市長パワハラ,「自身の言動がパワハラに該当するという点や、根底にある人権意識の欠如について、残念ながら自覚...,2026-07-30 21:12:20+09:00,https://mainichi.jp/articles/20260730/k00/00m/...,True,NaN
2,3,ポーランド東部の村にミサイルが着弾、ロシア製か 人的被害なし,ポーランド東部の領空に30日未明、未確認の飛行体が侵入し、農地に墜落した。ポーランドのトゥス...,2026-07-30 21:08:14+09:00,https://mainichi.jp/articles/20260730/k00/00m/...,False,NaN
3,4,経済プラス：国債売りが再燃 消費減税に成長投資…「積極財政」を市場は警戒,高市早苗首相が30日、飲食料品の消費減税を来年4月から2年間1％に引き下げると表明したことを...,2026-07-30 21:05:45+09:00,https://mainichi.jp/articles/20260730/k00/00m/...,True,NaN
4,5,高市首相「2年後に責任持って元に戻す」 消費減税巡り,高市早苗首相は30日、2027年4月から2年間限定で飲食料品の消費税を1％に引き下げることに...,2026-07-30 21:01:07+09:00,https://mainichi.jp/articles/20260730/k00/00m/...,False,NaN
...,...,...,...,...,...,...,...
9765,9766,佐賀東が逆転勝利で初の準々決勝進出を決める 10年ぶりの日本一を目指した富山第一は3回戦敗退,,NaT,https://mainichi.jp/articles/20240102/sck/00m/...,False,NaN
9766,9767,4ゴールで快勝した市立船橋が準々決勝へ進出決定 星稜は3回戦敗退,,NaT,https://mainichi.jp/articles/20240102/sck/00m/...,False,NaN
9767,9768,奥抜侃志、「歯痒い」日本代表デビューを経て決意を新たに…古巣・大宮への想いも,,NaT,https://mainichi.jp/articles/20240101/sck/00m/...,False,NaN
9768,9769,森保監督が 能登半島地震 に言及「被災状況が最小限に収まることを願っています」,,NaT,https://mainichi.jp/articles/20240101/sck/00m/...,False,NaN


In [7]:
output_path = Path("mainichi_熊本地震_20260727以降.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"保存しました: {output_path.resolve()}")


保存しました: /Users/tj/IdeaProjects/sample/get-news/mainichi_能登半島地震.csv
